# 🏋️ SFT on Qwen2.5 over Alpaca (laptop-friendly defaults)

This notebook is **domain-agnostic**. Set `DOMAIN_NAME` to whatever engagement you're working on (e.g. `asset_integrity`, `customer_grasps`, `ai_llm`, `legal_nda_review`, ...) and it will create the YAML if needed and train against it.

The template auto-resolves to Qwen ChatML from the base-model ID.

**Default is `Qwen/Qwen2.5-0.5B-Instruct` + 200 rows** so the run finishes on a MacBook in a few minutes. For a full engagement on Colab / Brev / an RTX box, set `BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'` and bump `max_samples` — the chat template is identical, only the weights change.

In [ ]:
# ── Environment bootstrap (Colab / Brev / local) ──
# Clones the repo into /content on Colab/Brev so `from app...`
# resolves. On a local checkout the repo already exists and the
# clone is skipped; we just chdir into it.
import os, sys, subprocess, pathlib

REPO_URL  = 'https://github.com/valonys/finetuningtraining.git'
REPO_NAME = 'finetuningtraining'
BRANCH    = 'develop'

def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

# Resolve a working repo root. Priority:
#   1. Walk up from the current dir looking for an `app/` folder
#      (works for any local checkout, regardless of cwd).
#   2. On Colab/Brev, clone the repo under /content.
def _find_repo_root() -> pathlib.Path | None:
    here = pathlib.Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'app' / '__init__.py').exists():
            return candidate
    return None

root = _find_repo_root()
if root is None:
    target = pathlib.Path('/content') if _in_colab() else pathlib.Path.cwd()
    dest = target / REPO_NAME
    if not dest.exists():
        print(f'📥 Cloning {REPO_URL} ({BRANCH}) -> {dest}')
        subprocess.check_call(
            ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(dest)]
        )
    root = dest

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print(f'✅ Repo root: {root}')

# Install the runtime deps. On Colab the base image has torch + transformers
# but not trl, peft, datasets at the right pins; a quiet pip run is cheap.
if _in_colab():
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'transformers>=4.44', 'trl>=0.11', 'peft>=0.12',
        'datasets>=2.20', 'accelerate>=0.33', 'pyyaml', 'pydantic>=2.0',
    ])
    print('✅ Dependencies installed')

In [ ]:

# ── Pick your domain name for this engagement ──
DOMAIN_NAME = 'ai_llm'   # ← change to your engagement, e.g. 'asset_integrity', 'customer_grasps'

# ── Base model ──
# Default is the 0.5B variant so the notebook is runnable on a laptop.
# Swap in 7B (or any other Qwen ChatML model) when you have the VRAM.
BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
# BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'    # Colab A100 / Brev L4+ / RTX 4090

from app.config_loader import create_domain_config, domain_config_exists, load_domain_config

if not domain_config_exists(DOMAIN_NAME):
    create_domain_config(
        name=DOMAIN_NAME,
        system_prompt='You are a senior AI engineer specialising in LLM post-training, evaluation, and efficient inference.',
        constitution=[
            'Cite primary sources for any non-obvious claim.',
            'Prefer reproducible benchmarks (MMLU, GSM8K, MT-Bench) with explicit hardware.',
            'Never hallucinate paper titles, arXiv IDs, or numbers.',
        ],
    )
    print(f'✅ Created configs/domains/{DOMAIN_NAME}.yaml')
else:
    print(f'✔️  Using existing configs/domains/{DOMAIN_NAME}.yaml')

config = load_domain_config(DOMAIN_NAME)
print(config['domain_name'], '—', config['system_prompt'][:80])

In [ ]:
from app.trainers import AgnosticSFTTrainer

# `max_samples` is now actually honored (bug fix: v3.0.1). 200 rows is
# enough to see the loss curve drop meaningfully on the 0.5B model.
# Bump to 5000+ on Colab/Brev for a real run.
MAX_SAMPLES = 200

trainer = AgnosticSFTTrainer(
    config=config,
    base_model_id=BASE_MODEL,
    hf_dataset_config={
        'repo_id': 'tatsu-lab/alpaca',
        'split': 'train',
        'input_column': 'instruction',
        'output_column': 'output',
        'max_samples': MAX_SAMPLES,
    },
)
result = trainer.train()
result